# Model Comparison with Permutation Tests and CDS

This notebook demonstrates how to:
1. Compare ECL across multiple models using the paired permutation test (Algorithm 5)
2. Decompose influence profiles using Context Decay Spectroscopy (CDS)

We use three synthetic models with different decay lengths to simulate short-, medium-, and long-context architectures.

In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import matplotlib.pyplot as plt

from ecl.models.base import SyntheticModel
from ecl.influence import compute_influence_profile
from ecl.ecl import ECL, cumulative_influence
from ecl.estimation import permutation_test
from ecl.cds import fit_cds, select_n_components
from ecl.perturbations import RandomSubstitution

## Setup: Three Models, Paired Loci

In [ ]:
SEQ_LEN, EMBED_DIM = 500, 32
N_LOCI, N_SEQS = 40, 10
rng = np.random.default_rng(42)
reference = SEQ_LEN // 2
perturbation = RandomSubstitution()

models = {
    "Short (CNN-like)": SyntheticModel(SEQ_LEN, EMBED_DIM, decay_length=20),
    "Medium (Transformer-like)": SyntheticModel(SEQ_LEN, EMBED_DIM, decay_length=80),
    "Long (SSM-like)": SyntheticModel(SEQ_LEN, EMBED_DIM, decay_length=180),
}

# Compute per-locus ECL for all models at matched loci
ecl_per_model = {name: np.zeros(N_LOCI) for name in models}

for j in range(N_LOCI):
    seqs = rng.integers(0, 4, size=(N_SEQS, SEQ_LEN)).astype(np.int8)
    for name, model in models.items():
        d, infl = compute_influence_profile(
            model, seqs, reference, max_distance=200,
            positions_per_distance=2, perturbation=perturbation,
            rng=rng, show_progress=False,
        )
        ecl_per_model[name][j] = ECL(d, infl, beta=0.9)

for name, vals in ecl_per_model.items():
    print(f"{name:30s}  mean ECL_0.9 = {vals.mean():.1f} +/- {vals.std():.1f} bp")

## Permutation Tests (Algorithm 5)

Test whether the ECL difference between each pair of models is statistically significant.

In [ ]:
model_names = list(models.keys())
print(f"{'Comparison':55s} {'Mean diff':>10s} {'p-value':>10s} {'95% CI':>15s}")
print("-" * 92)

for i in range(len(model_names)):
    for j in range(i + 1, len(model_names)):
        a, b = model_names[i], model_names[j]
        diff, p, ci = permutation_test(
            ecl_per_model[b], ecl_per_model[a], n_permutations=5000, rng=rng
        )
        label = f"{b} - {a}"
        print(f"{label:55s} {diff:10.1f} {p:10.4f} [{diff-ci:6.1f}, {diff+ci:6.1f}]")

## Context Decay Spectroscopy (CDS)

Decompose each model's influence profile into a mixture of exponentials: $I(d) \approx \sum_k a_k e^{-\lambda_k d}$.

In [ ]:
# Compute one representative profile per model
seqs = rng.integers(0, 4, size=(N_SEQS, SEQ_LEN)).astype(np.int8)
cds_results = {}

for name, model in models.items():
    d, infl = compute_influence_profile(
        model, seqs, reference, max_distance=200,
        positions_per_distance=5, perturbation=perturbation,
        rng=rng, show_progress=False,
    )
    best_K, results = select_n_components(d, infl, max_K=3)
    cds_results[name] = (d, infl, results[best_K - 1])
    print(f"\n{name} (best K={best_K}):")
    for k in range(best_K):
        a_k = results[best_K - 1]["amplitudes"][k]
        lam_k = results[best_K - 1]["decay_rates"][k]
        hl = 0.693 / lam_k if lam_k > 0 else float("inf")
        print(f"  Component {k+1}: amplitude={a_k:.4f}, decay_rate={lam_k:.4f}, half-life={hl:.0f} bp")

## Visualization

In [ ]:
colors = {"Short (CNN-like)": "#e41a1c", "Medium (Transformer-like)": "#377eb8", "Long (SSM-like)": "#4daf4a"}
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# A: Influence profiles with CDS fits
ax = axes[0]
for name, (d, infl, cds) in cds_results.items():
    ax.semilogy(d[1:], infl[1:], color=colors[name], alpha=0.4, linewidth=1)
    ax.semilogy(d[1:], cds["fitted"][1:], color=colors[name], linewidth=2, label=name)
ax.set_xlabel("Distance (bp)")
ax.set_ylabel("Influence energy")
ax.set_title("A. Influence Profiles + CDS Fits")
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

# B: Paired difference histogram
ax = axes[1]
diff = ecl_per_model["Long (SSM-like)"] - ecl_per_model["Short (CNN-like)"]
ax.hist(diff, bins=15, alpha=0.7, color="#377eb8", edgecolor="black")
ax.axvline(diff.mean(), color="red", linewidth=2, label=f"Mean = {diff.mean():.0f} bp")
ax.axvline(0, color="black", ls="--")
ax.set_xlabel("ECL_0.9 difference (Long - Short) [bp]")
ax.set_ylabel("Count")
ax.set_title("B. Paired Differences")
ax.legend()
ax.grid(True, alpha=0.3)

# C: ECL box plot
ax = axes[2]
model_names = list(models.keys())
data = [ecl_per_model[n] for n in model_names]
bp = ax.boxplot(data, labels=[n.split("(")[0].strip() for n in model_names], patch_artist=True)
for patch, name in zip(bp["boxes"], model_names):
    patch.set_facecolor(colors[name])
    patch.set_alpha(0.6)
ax.set_ylabel("ECL_0.9 (bp)")
ax.set_title("C. ECL Distribution by Model")
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()